# NB17 — Quantile Regression

> **Model the median (or any quantile) of y — not just the mean.**

OLS models E[y | x] — the conditional **mean**.
Quantile regression models Q_τ[y | x] — the conditional **τ-th quantile**.

- τ = 0.5 → median regression (robust to outliers)
- τ = 0.9 → "90th percentile of y given x"
- τ = 0.1 → "10th percentile"

Useful when:
- Data is skewed or heteroscedastic
- You need prediction intervals
- You care about tails (e.g. financial risk, SLAs)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import QuantileRegressor

np.random.seed(0)
n = 200
X = np.sort(np.random.uniform(0, 10, n))
# Heteroscedastic: variance grows with X
y = 2*X + 5 + np.random.normal(0, 0.5*X + 0.5, n)

X_2d   = X.reshape(-1, 1)
X_plot = np.linspace(0, 10, 200).reshape(-1, 1)

quantiles = [0.1, 0.25, 0.5, 0.75, 0.9]
colors     = ['#2196F3','#4CAF50','crimson','#FF9800','#9C27B0']

plt.figure(figsize=(10, 6))
plt.scatter(X, y, s=12, alpha=0.4, color='lightgray', zorder=1)

for q, color in zip(quantiles, colors):
    qr = QuantileRegressor(quantile=q, alpha=0, solver='highs').fit(X_2d, y)
    plt.plot(X_plot, qr.predict(X_plot), color=color, linewidth=2.5,
             label=f'Q={q}')

plt.xlabel('X'); plt.ylabel('y')
plt.title('Quantile regression: separate lines for each quantile')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print("Notice the lines fan out — variance increases with X (heteroscedasticity).")
print("OLS would give a single line, missing this structure entirely.")


In [ ]:
# Pinball loss — what quantile regression minimises
import numpy as np
import matplotlib.pyplot as plt

def pinball_loss(r, tau):
    return np.where(r >= 0, tau*r, (tau-1)*r)

r = np.linspace(-3, 3, 300)
fig, ax = plt.subplots(figsize=(8, 4))
for tau, color in [(0.1,'blue'), (0.5,'green'), (0.9,'red')]:
    ax.plot(r, pinball_loss(r, tau), color=color, linewidth=2, label=f'τ={tau}')
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('Residual r = y − ŷ'); ax.set_ylabel('Loss')
ax.set_title('Pinball loss: asymmetric penalty encourages the right quantile')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

print("τ=0.5 (median): symmetric → equal penalty for over- and under-prediction")
print("τ=0.9: heavy penalty for underestimation → model overshoots → 90th percentile")
print("τ=0.1: heavy penalty for overestimation → model undershoots → 10th percentile")


In [ ]:
# Prediction interval using quantile regression
import numpy as np
from sklearn.linear_model import QuantileRegressor
import matplotlib.pyplot as plt

qr_low  = QuantileRegressor(quantile=0.1, alpha=0, solver='highs').fit(X_2d, y)
qr_med  = QuantileRegressor(quantile=0.5, alpha=0, solver='highs').fit(X_2d, y)
qr_high = QuantileRegressor(quantile=0.9, alpha=0, solver='highs').fit(X_2d, y)

X_plot = np.linspace(0, 10, 200).reshape(-1,1)
low  = qr_low.predict(X_plot)
med  = qr_med.predict(X_plot)
high = qr_high.predict(X_plot)

plt.figure(figsize=(10, 5))
plt.scatter(X, y, s=12, alpha=0.3, color='steelblue')
plt.fill_between(X_plot.ravel(), low, high, alpha=0.2, color='orange', label='80% prediction band (Q0.1–Q0.9)')
plt.plot(X_plot, med, 'r-', linewidth=2, label='Median (Q0.5)')
plt.xlabel('X'); plt.ylabel('y')
plt.title('Prediction band from quantile regression')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


## Key Takeaways

| | OLS | Quantile (τ=0.5) |
|--|-----|-----------------|
| Models | Conditional mean | Conditional median |
| Sensitive to outliers | Yes | No |
| Loss function | Squared | Absolute (pinball) |
| Multiple lines | No | Yes (one per τ) |
| Prediction intervals | From normality assumption | Direct from Q0.1, Q0.9 |

**Next:** NB18 — Bayesian Linear Regression.
